# RFRecF — H&M Sampled Dataset

Communication-efficient extension of RFRec. Uses L2GD stochastic protocol:
with probability `threshold` clients do a local update; otherwise the server
aggregates item embeddings and clients step toward consensus via parameter `eta`.


In [12]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import math
import os
import itertools


## 1. Load Data


In [15]:
TARGET_USERS     = 1000
MIN_INTERACTIONS = 10
N_QUANTILES      = 10
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_INTERACTIONS}_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cache not found in '{SAMPLED_DATA_DIR}/'. Run sampling notebook first."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded : {cache_tag}")
print(f"Users  : {n_users} | Items : {n_items}")
print(f"Train  : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")


Loaded : u1000_m10_s42
Users  : 474 | Items : 5323
Train  : 15483 | Val : 3871 | Test : 7214


## 2. Build Rating Matrices


In [18]:
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())

def make_matrix(df, n_u, n_i, min_r, max_r):
    mat   = torch.full((n_u, n_i), -1.0)
    users = torch.tensor(df["customer_id"].values,  dtype=torch.long)
    items = torch.tensor(df["product_code"].values, dtype=torch.long)
    vals  = torch.tensor((df["bought"].values - min_r) / (max_r - min_r),
                         dtype=torch.float32)
    mat[users, items] = vals
    return mat

train_matrix = make_matrix(train_df, n_users, n_items, min_rating, max_rating)
val_matrix   = make_matrix(val_df,   n_users, n_items, min_rating, max_rating)
test_matrix  = make_matrix(test_df,  n_users, n_items, min_rating, max_rating)

print(f"Train : {(train_matrix != -1).sum().item()} observed")
print(f"Val   : {(val_matrix   != -1).sum().item()} observed")
print(f"Test  : {(test_matrix  != -1).sum().item()} observed")


Train : 15483 observed
Val   : 3871 observed
Test  : 7214 observed


## 3. Loss


In [ ]:
class RFRecFLoss(nn.Module):
    """MSE + L2 user reg + L2 item reg (no drift term — drift handled by L2GD step)."""
    def __init__(self, lam_u=0.1, lam_v=0.0):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v

    def forward(self, matrix, u_features, v_features):
        mask      = (matrix != -1).float()
        predicted = torch.sigmoid(torch.mm(u_features, v_features.t()))
        pred_err  = torch.sum(((matrix - predicted) ** 2) * mask)
        u_reg     = self.lam_u * torch.sum(u_features.norm(dim=1))
        v_reg     = self.lam_v * torch.sum(v_features.norm(dim=1))
        return pred_err + u_reg + v_reg, pred_err


def compute_rmse(matrix, predicted, min_r, max_r):
    mask    = (matrix != -1).float()
    n       = mask.sum()
    if n == 0: return float('nan')
    pred_sc = predicted * (max_r - min_r) + min_r
    true_sc = matrix    * (max_r - min_r) + min_r
    return math.sqrt((((pred_sc - true_sc) ** 2 * mask).sum() / n).item())

print("RFRecFLoss, compute_rmse defined.")


## 4. Parameter Tuning -- Grid Search

Searches `lr`, `lam_u`, `latent_vectors`. `threshold=0.8`, `eta=1` held fixed.
Run **once** after building matrices.


In [ ]:
LR_GRID     = [0.001, 0.01, 0.1]
LAM_GRID    = [0.01,  0.1,  0.3]
LATENT_GRID = [10,    20,   40 ]
TUNE_EPOCHS = 20
TUNE_PAT    = 5
TUNE_N_USERS = min(50, n_users)   # subset for speed

param_grid = list(itertools.product(LR_GRID, LAM_GRID, LATENT_GRID))
print(f"Grid search: {len(param_grid)} combinations x {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'lr':>7}  {'lam':>6}  {'K':>4}  {'best val RMSE':>14}")
print("-" * 42)
grid_results = []
THRESHOLD = 0.8
ETA       = 1.0
tune_train = train_matrix[:TUNE_N_USERS]
tune_val   = val_matrix[:TUNE_N_USERS]
NC_T = TUNE_N_USERS

for _k, (_lr, _lam, _K) in enumerate(param_grid, 1):
    torch.manual_seed(0)
    _ufs = [torch.randn(1, _K, requires_grad=True) for _ in range(NC_T)]
    _vfs = [torch.randn(n_items, _K, requires_grad=True) for _ in range(NC_T)]
    for _uf, _vf in zip(_ufs, _vfs):
        _uf.data.mul_(0.01); _vf.data.mul_(0.01)
    _loss_fn = RFRecFLoss(lam_u=_lam, lam_v=0.0)
    _opts    = [torch.optim.Adam([_ufs[i], _vfs[i]], lr=_lr) for i in range(NC_T)]
    _aver    = sum(_vfs[i].data for i in range(NC_T)) / NC_T
    _best_val, _wait = float('inf'), 0
    for _ep in range(TUNE_EPOCHS):
        p = torch.rand(1).item()
        if p > THRESHOLD:
            for i in range(NC_T):
                _opts[i].zero_grad()
                _loss, _ = _loss_fn(tune_train[i:i+1], _ufs[i], _vfs[i])
                _loss.backward()
                _opts[i].step()
        else:
            _tmp = torch.zeros(n_items, _K)
            with torch.no_grad():
                for i in range(NC_T): _tmp += _vfs[i].data
                _aver = _tmp / NC_T
                for i in range(NC_T):
                    _vfs[i].data += ETA * (_aver - _vfs[i].data)
        with torch.no_grad():
            _pred = torch.cat(
                [torch.sigmoid(torch.mm(_ufs[i], _vfs[i].t())) for i in range(NC_T)], dim=0)
            _v = compute_rmse(tune_val, _pred, min_rating, max_rating)
        if _v < _best_val: _best_val, _wait = _v, 0
        else:
            _wait += 1
            if _wait >= TUNE_PAT: break
    grid_results.append((_best_val, _lr, _lam, _K))
    _marker = ' <--' if _best_val == min(r[0] for r in grid_results) else ''
    print(f"{_k:>3}  {_lr:>7.4f}  {_lam:>6.3f}  {_K:>4d}  {_best_val:>14.4f}{_marker}")
best_val_t, best_lr, best_lam, best_K = min(grid_results, key=lambda x: x[0])
print(f"\nBest val RMSE  : {best_val_t:.4f}")
print(f"  lr             = {best_lr}")
print(f"  lam            = {best_lam}")
print(f"  latent_vectors = {best_K}")
lr             = best_lr
lam            = best_lam
latent_vectors = best_K
print("\nHyperparameters updated. Run the training cell.")


## 5. Training


In [ ]:
try: latent_vectors
except NameError: latent_vectors = 20
try: lr
except NameError: lr = 0.1
try: lam
except NameError: lam = 0.1
num_epoch   = 200
patience    = 10
NUM_CLIENTS = n_users
THRESHOLD   = 0.8
ETA         = 1.0

torch.manual_seed(0)
user_features = [torch.randn(1, latent_vectors, requires_grad=True) for _ in range(NUM_CLIENTS)]
item_features = [torch.randn(n_items, latent_vectors, requires_grad=True) for _ in range(NUM_CLIENTS)]
for i in range(NUM_CLIENTS):
    user_features[i].data.mul_(0.01)
    item_features[i].data.mul_(0.01)

loss_fn    = RFRecFLoss(lam_u=lam, lam_v=0.0)
optimizers = [torch.optim.Adam([user_features[i], item_features[i]], lr=lr)
              for i in range(NUM_CLIENTS)]

best_val_rmse, patience_count = float('inf'), 0
best_uf, best_vf = None, None

for epoch in range(num_epoch):
    p = torch.rand(1).item()
    if p > THRESHOLD:
        for i in range(NUM_CLIENTS):
            optimizers[i].zero_grad()
            loss, _ = loss_fn(train_matrix[i:i+1], user_features[i], item_features[i])
            loss.backward()
            optimizers[i].step()
    else:
        tmp = torch.zeros(n_items, latent_vectors)
        with torch.no_grad():
            for i in range(NUM_CLIENTS): tmp += item_features[i].data
            aver_item = tmp / NUM_CLIENTS
            for i in range(NUM_CLIENTS):
                item_features[i].data += ETA * (aver_item - item_features[i].data)

    with torch.no_grad():
        pred_parts = [torch.sigmoid(torch.mm(user_features[i], item_features[i].t()))
                      for i in range(NUM_CLIENTS)]
        pred_full  = torch.cat(pred_parts, dim=0)
        train_rmse = compute_rmse(train_matrix, pred_full, min_rating, max_rating)
        val_rmse   = compute_rmse(val_matrix,   pred_full, min_rating, max_rating)

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | train RMSE: {train_rmse:.4f} | val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_uf = [user_features[i].data.clone() for i in range(NUM_CLIENTS)]
        best_vf = [item_features[i].data.clone() for i in range(NUM_CLIENTS)]
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

with torch.no_grad():
    for i in range(NUM_CLIENTS):
        user_features[i].data.copy_(best_uf[i])
        item_features[i].data.copy_(best_vf[i])
print(f"\nBest val RMSE: {best_val_rmse:.4f}")


## 6. Test Evaluation


In [ ]:
with torch.no_grad():
    pred_parts = [torch.sigmoid(torch.mm(user_features[i], item_features[i].t()))
                  for i in range(NUM_CLIENTS)]
    pred_test  = torch.cat(pred_parts, dim=0)
    test_rmse  = compute_rmse(test_matrix, pred_test, min_rating, max_rating)
    mask     = (test_matrix != -1).float()
    n_obs    = mask.sum()
    pred_sc  = pred_test   * (max_rating - min_rating) + min_rating
    true_sc  = test_matrix * (max_rating - min_rating) + min_rating
    test_mae = (torch.abs(pred_sc - true_sc) * mask).sum() / n_obs
print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae.item():.4f}")
